# NDVI (HLS Landsat/Sentinel-2) per ISMN Station

<br>

For each station, pulls NDVI from NASA's **HLS** (Harmonized Landsat
Sentinel-2) collections -- **HLSL30**/**HLSS30** -- at the SAR acquisition
dates already saved by [`01b_get_nisar_gslc_windows.ipynb`](01b_get_nisar_gslc_windows.ipynb).

NDVI values are **not needed for the estimations** [`03a_estimate_sm_nisar.ipynb`/`03b_estimate_sm_s1.ipynb`], but for the exploratory analysis of `04_station_overview.ipynb`.

Each HLS band is its own Cloud-Optimized GeoTIFF, so a windowed `rasterio`
read over HTTPS pulls only the pixels under an 80x80m station window via a
couple of HTTP range requests -- no full-granule download needed.

**Target dates, not a continuous time series** -- NDVI is only pulled for
the SAR acquisition dates already on disk (via `load_station_acquisitions`,
the same loader `02b_calc_nisar_coherence_closure.ipynb` uses), matched to
the nearest HLS scene within `MAX_GAP_DAYS`.

Implemented by [`util/hls_ndvi.py`](../util/hls_ndvi.py).

:::{note} Requires 01b_get_nisar_gslc_windows.ipynb to have already been run
Target dates come from `nisar_data/alaska_GSLC_windows/*.nc` -- if that
directory is empty or missing, every station will show 0 target dates.
:::

<hr>

## Overview

1. Load station locations
2. Get target (SAR acquisition) dates per station from the saved GSLC windows
3. Sanity-check HLS band/Fmask structure against real granules
4. Process each station: search HLSL30+HLSS30, match dates, compute window NDVI, save CSV
5. (Optional) Preview one station's output

<hr>

In [ ]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime, timezone

import earthaccess

from util.nisar_station_timeseries import get_fsspec_https_session_with_timeout
from util.nisar_coherence_closure import load_station_acquisitions
from util.hls_ndvi import (
    inspect_hls_structure,
    search_hls_for_station,
    process_station_ndvi,
    DEFAULT_WINDOW_FOOTPRINT_M,
    DEFAULT_MAX_GAP_DAYS,
)

## 1. Load station locations

Same station list as the other notebooks in this cookbook.

In [ ]:
stations_path = Path.cwd() / "ismn_data" / "alaska" / "stations.geojson"

with open(stations_path) as f:
    stations_geojson = json.load(f)

stations = [
    {
        "name": feat["properties"]["station_val"],
        "lon": feat["geometry"]["coordinates"][0],
        "lat": feat["geometry"]["coordinates"][1],
    }
    for feat in stations_geojson["features"]
]

print(f"Loaded {len(stations)} stations")
stations[:3]

## 2. Target dates per station

Every SAR acquisition date already saved for a station, across all of its
kept tracks -- NDVI only cares about the calendar date, not which NISAR
track it came from.

In [ ]:
gslc_dir = Path.cwd() / "nisar_data" / "alaska_GSLC_windows"

target_dates_by_station = {}
for station in stations:
    groups = load_station_acquisitions(gslc_dir, station["name"])
    dates = sorted({acq["datetime"] for track_acqs in groups.values() for acq in track_acqs})
    target_dates_by_station[station["name"]] = dates
    print(f"{station['name']}: {len(dates)} target date(s)")

## 3. Sanity-check HLS band/Fmask structure against real granules

Opens one real HLSL30 and one real HLSS30 granule and prints their asset
URLs plus Red/NIR/Fmask shape/dtype/CRS, to check `util/hls_ndvi.py`'s
layout assumptions against real data. Band-name suffixes and short names
are already confirmed against a live CMR search; the Fmask **bit layout**
is not -- check the printed value distribution before trusting the cloud
mask across every station.

In [ ]:
earthaccess.login()

fs = get_fsspec_https_session_with_timeout()

TEMPORAL = ("2026-06-01", datetime.now(timezone.utc).strftime("%Y-%m-%d"))

_sanity_results = search_hls_for_station(stations[0], TEMPORAL)
print(f"Found {len(_sanity_results)} candidate HLS granule(s) for station '{stations[0]['name']}'")

_seen_collections = set()
for _r in _sanity_results:
    _native_id = _r["meta"]["native-id"]
    _collection = _native_id.split(".")[1]  # "L30" or "S30"
    if _collection in _seen_collections:
        continue
    _seen_collections.add(_collection)
    print(f"\n{'=' * 70}\n{_native_id}\n{'=' * 70}")
    inspect_hls_structure(_r, fs)

## 4. Process each station: search, match dates, compute NDVI, save CSV

For each station: search `HLSL30`+`HLSS30` over the full date range; for
each target date (Section 2), find the nearest available HLS scene
(recorded even if the gap exceeds `MAX_GAP_DAYS`, with `gap_days` showing
how far off); read the Red/NIR/Fmask bands as an 80x80m window; mask
cloud/shadow/snow-ice pixels and the reflectance fill value, then compute
NDVI = (NIR-Red)/(NIR+Red) over whatever valid pixels remain; save
`{station_name}_HLS_ndvi.csv` with `n_valid_px`/`n_total_px` so a
heavily-masked window is visible rather than silently averaged over 1-2
pixels.

In [ ]:
WINDOW_FOOTPRINT_M = DEFAULT_WINDOW_FOOTPRINT_M  # 80m, matches the GSLC/Sentinel-1 window
MAX_GAP_DAYS = DEFAULT_MAX_GAP_DAYS

output_dir = Path.cwd() / "nisar_data" / "alaska_HLS_ndvi"

for station in stations:
    target_dates = target_dates_by_station[station["name"]]
    if not target_dates:
        print(f"{station['name']}: no target dates, skipping")
        continue

    search_results = search_hls_for_station(station, TEMPORAL)
    csv_path = process_station_ndvi(
        station, target_dates, search_results, output_dir, fs,
        window_footprint_m=WINDOW_FOOTPRINT_M, max_gap_days=MAX_GAP_DAYS,
    )
    print(f"{station['name']}: {len(search_results)} HLS scene(s) found, "
          f"{len(target_dates)} target date(s) -> {csv_path.name}")

## 5. (Optional) Preview one station's output

In [ ]:
import pandas as pd

preview_station = stations[0]["name"]
ndvi_df = pd.read_csv(output_dir / f"{preview_station}_HLS_ndvi.csv")
ndvi_df

<hr>

## Summary

You now have, under `nisar_data/alaska_HLS_ndvi/`, one CSV per station:
`target_date`, `hls_date`, `sensor`, `gap_days`, `ndvi_mean`, `ndvi_std`,
`n_valid_px`, `n_total_px`.

Join on `target_date` against any SM-inversion output using the same
acquisition dates to plot NDVI alongside inversion residuals. A row with
`n_valid_px` close to 0 or a large `gap_days` should be treated as
unreliable rather than a genuine vegetation signal.

Safe to re-run any time -- both CSVs are fully rebuilt from scratch each
run.

<hr>